# Topic 4: LCEL -- the `|` chain (the big leap)

You have already used LCEL without naming it: `(prompt_a | llm | parser).invoke(...)` in Topic 2.
LCEL = **LangChain Expression Language**, and the `|` operator is its heart.

## Foundation: what a "chain" actually is

Think of a **pipe**. Data enters the left side, gets passed through each stage to the right, and the final result comes out the right end.

```
   input            input/output flow                 output
   -----            -----------------                 ------
  question ---> retriever ---> prompt ---> llm ---> parser ---> str
```

**The only rule to internalize:** `A | B` works when `A.invoke(x)` produces something that `B` can accept. The left stage *output shape* must match the right stage *input shape*. That is the whole contract.

Two consequences follow:
- Strings flow easily: `prompt | llm | parser` works because prompt gets a dict, llm gets a string, parser gets an `AIMessage`.
- The **prompt is the shape-shifter** -- it is the only stage that transforms a dict `{context, question}` into a single string.

## The one question to keep in mind

> If someone asks "what is paranoid about?", how does the LLM produce an answer?

The answer: **four things must happen every time** --
1. find the right chunks,
2. stuff them (and the question) into a prompt,
3. send that prompt to the LLM,
4. get the answer back as plain text.

The code below builds that, **one testable step at a time**, so you can watch the data flow.

## The shape evolution in a RAG chain

| Stage | Input | Output |
|---|---|---|
| retriever | str (question) | `list[Document]` (top-k chunks) |
| format_docs | `list[Document]` | str (chunks joined by blank lines) |
| prompt | dict `{context, question}` | str (whole filled prompt) |
| llm | str (the prompt) | `AIMessage` |
| parser | `AIMessage` | str (just the answer text) |

`format_docs` is a small plain function you write yourself -- it is exactly the missing shape that connects retriever output to the prompt's `{context}` slot.

## The two helpers you must know

**1. `RunnablePassthrough` -- "echo input unchanged."** It is used for dict *merging*:

```python
chain = (
    {
        "context":  retriever | format_docs,     # str -> list[Document] -> str
        "question": RunnablePassthrough(),        # str input passed through as-is
    }
    | prompt
    | llm
    | parser
)
```

The input dict `{"question": "..."}` gets split:
- the `question` key routes to `RunnablePassthrough()` (echoes the string),
- the `context` key routes through `retriever | format_docs`,
- the prompt receives a merged dict `{context, question}`.

**2. Grounding.** The `|` by itself is not RAG -- you build RAG by defining *what feeds each slot*.

## The full chain, run live on eve

- In-scope question -> answer grounded in retrieved lyrics.
- Out-of-scope question -> hits the `"I don't know"` guard.

## Honest caveat (the "Black Sabbath moment")

In the live run, the in-scope answer said *"Paranoid" by Black Sabbath* -- but "Black Sabbath"
was NOT in the retrieved chunks. That is the model **leaking prior knowledge** on top of
grounded context. RAG grounds but does not *erase* the model's memory. Detecting exactly this
is the job of **Topic 5 (provenance)**.

## Topic 4 complete -- the summary

- `|` joins stages left->right; data flows through each. **The only rule:** left output shape must feed right input shape.
- Shape evolution: `str -> list[Document] -> str -> str`.
- The **prompt is the shape-shifter** -- it turns your input dict into one string the LLM reads.
- `RunnablePassthrough()` echoes input unchanged; it lets the retriever branch off to build `context` while the raw question flows through as `question`.
- Full chain: `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser`.
- Verified live: grounded in-scope answer, and the "I don't know" guard for out-of-scope.
- Grounded context != perfect truth -- the model can still embellish. That is Topic 5.

## The step-by-step walkthrough (run these cells in order)

Each step below shows one piece of the chain and its actual input/output type.
Together they are the full RAG chain. Watch the data flow at every stage.

In [2]:
# ---- STEP 1: build the store (the chunk index) ----
# You did this in Step 4 of the 4-step pipeline. Nothing new here.
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# robust path finding: works no matter which folder Jupyter opened the notebook in
def find_album():
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for cand in (
            root / "data_ingestion" / "heylog_eve_album.txt",
            root / "section_5_LangChain" / "data_ingestion" / "heylog_eve_album.txt",
        ):
            if cand.exists():
                return cand
    raise FileNotFoundError("heylog_eve_album.txt not found")

emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(find_album())).load())
store = FAISS.from_documents(chunks, emb)
print("STEP 1: store built, chunks:", store.index.ntotal)

STEP 1: store built, chunks: 51


In [20]:
# ---- STEP 2: the retriever returns a LIST of Documents ----
retriever = store.as_retriever(search_kwargs={"k": 5})
hits = retriever.invoke("what is 12 gauge about?")

print("STEP 2: retriever.invoke() ->", type(hits).__name__, "| len:", len(hits))
for i, d in enumerate(hits):
    print(f"  [{i}]", repr(d.page_content))

STEP 2: retriever.invoke() -> list | len: 5
  [0] 'INTERPRETATION:\nThe centerpiece. About a relationship where she keeps borrowing\nhis wardrobe until slowly, almost invisibly, she has his whole\nattire - everything he owns. The become metaphors for his\nidentity slowly being taken over. He pretends not to care about\n"us" but admits he is "struggling to exist." The locked-up\n12 gauge (shotgun) represents a destructive urge kept suppressed'
  [1] "============================================================\nTRACK 5: 12 GAUGE\n============================================================\nLYRICS:\n[Verse 1]\n(Ooh)\nMovin' swift, movin' clean\nAll my thrift big on me\nAnd I look goofy when I see (Ooh)\nOrder double in XL when my body lean as hell\nAnd it's all because I'm— (Well, ooh)\nMaybe I should get clothes that I fit for my size"
  [2] 'INTERPRETATION:\nA song about exhaustion and pushing past your limits. Every\nmuscle group feels torn, skin turning blue, the wind itself\nfeels 

In [21]:
# ---- STEP 3: format_docs turns the list into ONE string ----
# The LLM cannot read a list of Document objects -- it reads a string.
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

joined = format_docs(retriever.invoke("what is 12 gauge about?"))
print("STEP 3: format_docs ->", type(joined).__name__, "| chars:", len(joined))
print("  --- the joined string ---")
print(joined)

STEP 3: format_docs -> str | chars: 1587
  --- the joined string ---
INTERPRETATION:
The centerpiece. About a relationship where she keeps borrowing
his wardrobe until slowly, almost invisibly, she has his whole
attire - everything he owns. The become metaphors for his
identity slowly being taken over. He pretends not to care about
"us" but admits he is "struggling to exist." The locked-up
12 gauge (shotgun) represents a destructive urge kept suppressed

TRACK 5: 12 GAUGE
LYRICS:
[Verse 1]
(Ooh)
Movin' swift, movin' clean
All my thrift big on me
And I look goofy when I see (Ooh)
Order double in XL when my body lean as hell
And it's all because I'm— (Well, ooh)
Maybe I should get clothes that I fit for my size

INTERPRETATION:
A song about exhaustion and pushing past your limits. Every
muscle group feels torn, skin turning blue, the wind itself
feels like a physical punch. It is both a literal struggle
against a storm and a metaphor for burnout and endurance -
running head down, straigh

In [22]:
# ---- STEP 4: the prompt template has TWO holes ----
prompt = PromptTemplate.from_template(
    "Answer using ONLY the context. If absent, say \"I don't know.\"\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\nAnswer:"
)
filled = prompt.format(
    context="Track 7: PARANOID\nand his paranoia spirals",
    question="what is paranoid about?"
)
print("STEP 4: prompt holes:", prompt.input_variables)
print("  --- filled prompt ---")
print(filled)

STEP 4: prompt holes: ['context', 'question']
  --- filled prompt ---
Answer using ONLY the context. If absent, say "I don't know."

Context:
Track 7: PARANOID
and his paranoia spirals

Question: what is paranoid about?
Answer:


In [23]:
# ---- STEP 5: RunnablePassthrough echoes input UNCHANGED ----
# It does NOTHING -- deliberately. That is the point.
passthrough = RunnablePassthrough()
print("STEP 5: passthrough does nothing on purpose")
print("  'hello'  ->", passthrough.invoke("hello"))
print("  42       ->", passthrough.invoke(42))
print("  [1,2,3]  ->", passthrough.invoke([1,2,3]))

STEP 5: passthrough does nothing on purpose
  'hello'  -> hello
  42       -> 42
  [1,2,3]  -> [1, 2, 3]


In [24]:
# ---- STEP 6: the dict MERGES two paths into one dict ----
# One string question goes in. The dict fills the prompt's two holes:
#   - "context"  is built by finding chunks and joining them
#   - "question" is the raw question echoed through
# (No LLM yet -- look at what the prompt receives.)
llm = ChatOllama(model="mistral:7b")
parser = StrOutputParser()

chain_half = (
    {
        "context":  retriever | format_docs,   # question -> chunks -> joined text
        "question": RunnablePassthrough(),      # question -> echo unchanged
    }
    | prompt
)
print("STEP 6: dict merge -> filled prompt (no LLM yet)")
print(chain_half.invoke("what is paranoid about?"))

STEP 6: dict merge -> filled prompt (no LLM yet)
text='Answer using ONLY the context. If absent, say "I don\'t know."\n\nContext:\nINTERPRETATION:\nThe album\'s closing statement. An idyllic forest hike that\nsours into psychological unease. The narrator seeks refuge in\nnature to escape people, expectations, and his own relentless\nthoughts - wanting to live off-grid in total isolation. But\nhe is ashamed of merely hiding instead of facing his problems,\nand his paranoia spirals: deer that won\'t run, branches cracking,\n\n============================================================\nTRACK 7: PARANOID\n============================================================\nLYRICS:\n[Intro]\n(heylog)\n(Ooh-oh, oh)\n(Ooh-oh)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\n\n[Verse 2]\nHanging out down by the creek, couple deer following me\nIf I stop to look at them, they just stand there as if they freeze\nWhy haven\'t they ran away? Figur

In [25]:
# ---- STEP 7: THE FULL RAG CHAIN ----
# Now add the LLM (generate) and the parser (unwrap to string).
chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | parser
)
print("STEP 7: real chain")
print("Q: what is the song 12 Gauge about?")
print("A:", chain.invoke("what is the song 12 gauge about?"))
print()
print("Q: what color is the sofa in the album?")
print("A:", chain.invoke("what color is the sofa in the album?"))

STEP 7: real chain
Q: what is the song 12 Gauge about?
A:  The song 12 Gauge is about a person finding solace and escapism in nature, away from reality and its troubles. It also suggests feelings of shame, hurt, and a sense of being someone else. The locked-up 12 gauge shotgun could symbolize a destructive urge or stress that the individual keeps suppressed during their escape into nature.

Q: what color is the sofa in the album?
A:  I don't know. The context does not provide information about the color of the sofa in the album.


In [26]:
# ---- STEP 8: bonus -- .stream() on the whole chain (typing effect) ----
print("streaming the chain:")
for token in chain.stream("what is the song paranoid about?"):
    print(token, end="", flush=True)
print()

streaming the chain:
 The song "Paranoid" is about a narrator who, while seeking refuge in nature to escape people, expectations, and his own relentless thoughts, experiences psychological unease and paranoia. He encounters deer that won't run, branches cracking behind him, and feels as if he may be a schizophrenic, fearing for his life. This suggests that the song is primarily about feelings of paranoia and fear arising from isolation in nature.


In [27]:
# ---- STEP 8.2: bonus -- .stream() on the whole chain (typing effect) ----
print("streaming the chain:")
for token in chain.stream("what is the song cat claws about?"):
    print(token, end="", flush=True)
print()

streaming the chain:
 The song "Cat Claws" is not explicitly mentioned in the provided context.


In [28]:
# ---- STEP 8.3: bonus -- .stream() on the whole chain (typing effect) ----
print("streaming the chain:")
for token in chain.stream("what is the context u have?"):
    print(token, end="", flush=True)
print()

streaming the chain:
 The context provided here appears to be a song lyrics about a relationship, with themes of identity loss, conflict, and possession. The lyrics suggest that one person (the speaker) feels their partner has taken over their wardrobe and, by extension, their identity, causing tension and pain in the relationship. The speaker expresses feelings of confusion, frustration, and a struggle to maintain individuality while the other party seems unapologetic or indifferent. Additionally, there's a reference to a locked-up shotgun as a symbol for suppressed anger or destructive urges.


In [29]:
chain.invoke("What is the whole context about?")

" The context is about a deteriorating relationship where one person (the narrator) feels their identity is being taken over by the other. This is portrayed as a metaphorical borrowing of the other's wardrobe. The relationship is strained, causing the narrator distress and a suppressed destructive urge. The narrator also experiences feelings of solitude and exile, finding refuge in driving alone at night on empty roads."

In [ ]:
chain.invoke(str(input("Ask a question")))

In [16]:
chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f06d812bb10>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer using ONLY the context. If absent, say "I don\'t know."\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, model='mistral:7b')
| StrOutputParser()

In [17]:
def debug_retriever(question):
    docs = retriever.invoke(question)
    print(f"\n--- Retrieved {len(docs)} docs ---")
    for i, doc in enumerate(docs):
        print(f"[{i}] {doc.page_content[:200]}...")
    return format_docs(docs)

In [19]:
debug_retriever("name the song names in eve album")


--- Retrieved 3 docs ---
[0] [Outro]
Oh, oh-oh, oh
Oh, oh-oh, oh
(Like Adam and Eve)
Oh, oh-oh, oh (Eve, Eve)...
[1] ============================================================
TRACK 7: PARANOID
LYRICS:
[Intro]
(heylog)
(Ooh-oh, oh)
(Ooh-oh)
Oh, oh-oh, oh...
[2] HEYLOG - EVE (FULL ALBUM, 2024)
Debut album by heylog. 7 tracks, 25 minutes. Alt-pop / emo / independent.
The album is approached biblically, focused on the figure of...


'[Outro]\nOh, oh-oh, oh\nOh, oh-oh, oh\n(Like Adam and Eve)\nOh, oh-oh, oh (Eve, Eve)\n\n============================================================\nTRACK 7: PARANOID\n============================================================\nLYRICS:\n[Intro]\n(heylog)\n(Ooh-oh, oh)\n(Ooh-oh)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\nOh, oh-oh, oh (Eve, Eve)\n\nHEYLOG - EVE (FULL ALBUM, 2024)\n=================================\nDebut album by heylog. 7 tracks, 25 minutes. Alt-pop / emo / independent.\nThe album is approached biblically, focused on the figure of Eve.\n\n============================================================\nTRACK 1: GRAVEL\n============================================================\nLYRICS:\n[Instrumental]\n[Outro] (heylog)'